# Baseline Model Development

This notebook trains and compares several baseline classification models for obesity-risk prediction.

## Objectives

- Recreate the stratified training, validation, and test datasets
- Use the reusable preprocessing module
- Establish a simple benchmark
- Train multiple classification algorithms
- Evaluate models using consistent metrics
- Select promising models for hyperparameter tuning
- Keep the test dataset untouched until final model selection

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
CURRENT_PATH = Path.cwd()

possible_roots = [
    CURRENT_PATH,
    *CURRENT_PATH.parents,
]

PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "src" / "preprocessing.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

project_root_string = str(PROJECT_ROOT)

if project_root_string not in sys.path:
    sys.path.insert(0, project_root_string)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "obesity.csv"
)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

Project root: c:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System
Dataset exists: True


In [3]:
from src.preprocessing import (
    PREDICTIVE_FEATURES,
    build_preprocessor,
)

print(
    "Configured predictive features:",
    len(PREDICTIVE_FEATURES),
)

test_preprocessor = build_preprocessor()

print(
    "Preprocessor type:",
    type(test_preprocessor).__name__,
)

Configured predictive features: 16
Preprocessor type: ColumnTransformer


In [4]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (20758, 18)


,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [5]:
IDENTIFIER_COLUMN = "id"
TARGET_COLUMN = "NObeyesdad"
RANDOM_STATE = 42

In [6]:
X = df[PREDICTIVE_FEATURES].copy()
y = df[TARGET_COLUMN].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (20758, 16)
Target shape: (20758,)


In [7]:
missing_features = (
    set(PREDICTIVE_FEATURES)
    - set(df.columns)
)

unexpected_features = (
    set(X.columns)
    - set(PREDICTIVE_FEATURES)
)

print("Missing features:", missing_features)
print("Unexpected features:", unexpected_features)

Missing features: set()
Unexpected features: set()


In [8]:
assert not missing_features
assert not unexpected_features
assert list(X.columns) == PREDICTIVE_FEATURES

print("Feature configuration validation passed")

Feature configuration validation passed


In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_validation, X_test, y_validation, y_test = (
    train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )
)

In [10]:
split_summary = pd.DataFrame(
    {
        "Dataset": [
            "Training",
            "Validation",
            "Test",
        ],
        "Records": [
            len(X_train),
            len(X_validation),
            len(X_test),
        ],
        "Percentage": [
            len(X_train) / len(X) * 100,
            len(X_validation) / len(X) * 100,
            len(X_test) / len(X) * 100,
        ],
    }
)

split_summary["Percentage"] = (
    split_summary["Percentage"].round(2)
)

split_summary

,Dataset,Records,Percentage
0,Training,14530,70.0
1,Validation,3114,15.0
2,Test,3114,15.0


In [11]:
train_validation_overlap = set(
    X_train.index
) & set(X_validation.index)

train_test_overlap = set(
    X_train.index
) & set(X_test.index)

validation_test_overlap = set(
    X_validation.index
) & set(X_test.index)

print(
    "Training-validation overlap:",
    len(train_validation_overlap),
)

print(
    "Training-test overlap:",
    len(train_test_overlap),
)

print(
    "Validation-test overlap:",
    len(validation_test_overlap),
)

Training-validation overlap: 0
Training-test overlap: 0
Validation-test overlap: 0


In [12]:
assert X_train.shape == (14530, 16)
assert X_validation.shape == (3114, 16)
assert X_test.shape == (3114, 16)

assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

assert not train_validation_overlap
assert not train_test_overlap
assert not validation_test_overlap

assert (
    len(X_train)
    + len(X_validation)
    + len(X_test)
    == len(X)
)

print(
    "Baseline modelling dataset preparation passed"
)

Baseline modelling dataset preparation passed
